<a href="https://colab.research.google.com/github/Mar-SHieee/Customer-Rating-Prediction/blob/main/Customer_Rating_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.Title & Info

**Name:** Mahmoud Elaraby  
**Model:** Model 3 – Customer Rating Prediction (Regression)  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist:"https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce"

## 2. Business Understanding

## Domain

The domain of this project is **E-Commerce**.

The dataset comes from Olist, which is an online marketplace. Customers can buy products from different sellers and then give a review score after receiving their orders.

Customer ratings are important because they can show the customer's satisfaction with the product, delivery, and overall shopping experience.

---

## Model's Problem

The problem of this model is to **predict the customer review score** for an order.

The target variable is:

`review_score`

The review score ranges from **1 to 5**, where:

- 1 = Very Low Rating
- 2 = Low Rating
- 3 = Average Rating
- 4 = Good Rating
- 5 = Excellent Rating

This is a **Regression problem** because the model predicts a numerical value.

The model will use information related to the order, product, payment, seller, and delivery to predict the customer rating.

---

## Why It Matters

Predicting customer ratings can help an e-commerce company understand the factors that affect customer satisfaction.

For example, low ratings may be related to:

- Late delivery
- Long delivery time
- High freight cost
- Product category
- Seller performance

By predicting ratings, the company can identify possible problems and take action before more customers have a negative experience.

This can help improve:

- Customer satisfaction
- Delivery performance
- Seller performance
- Product quality monitoring
- Overall customer experience

---

## Success Criteria

The model will be considered successful if it can predict customer ratings with low prediction errors.

The models will be evaluated using:

- **MAE (Mean Absolute Error)** – lower is better.
- **RMSE (Root Mean Squared Error)** – lower is better.
- **R² Score** – higher is better.

Four regression models will be trained and compared:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. K-Nearest Neighbors Regressor

The best model will be selected based mainly on its performance using **MAE, RMSE, and R² Score**, and then the best model will be tuned using **GridSearchCV**.

## 3. Imports

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder , LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)



## 4. Load Data

In [10]:
customers = pd.read_csv("data/olist_customers_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
category_translation = pd.read_csv("data/product_category_name_translation.csv")

# Merge Orders with Customers
df = orders.merge(customers,on="customer_id",how="left")

# Merge with Order Items
df = df.merge(order_items,on="order_id",how="left")

# Merge with Products
df = df.merge(products,on="product_id",how="left")

# Merge with Sellers
df = df.merge(sellers,on="seller_id",how="left")

# Merge with Payments
df = df.merge(order_payments,on="order_id",how="left")

# Merge with Reviews
df = df.merge(order_reviews,on="order_id",how="left")

# Merge with category translation
df = df.merge(category_translation,on="product_category_name",how="left")

raw_orders = df.copy()

print('Orders:', orders.shape)
print('Customers:', customers.shape)
print('Items:', order_items.shape)
print('Payments:',order_payments.shape)
print('Products:', products.shape)
print('Sellers:', sellers.shape)
print('Reviews:', order_reviews.shape)
print('full Dataset',df.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'data/olist_customers_dataset.csv'

## 5. Data Audit

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
pd.set_option('display.max_columns', None)
df.tail(10)

In [ ]:
missing = pd.DataFrame({
    'count': df.isna().sum(),
    'percent': df.isna().mean().mul(100).round(2)
}).sort_values('percent', ascending=False).sort_values('count', ascending=False)

print('Missing values')
missing


In [ ]:
print('Exact duplicate rows:',df.duplicated().sum())
print('Duplicate Order_ID rows:',df.duplicated('order_id', keep=False).sum())

## Hidden nulls problem
there is no hidden nulls in the data as it didn't change after the code

In [ ]:
# Detect hidden/misleading null values
hidden_nulls = ['NA', 'N/A', 'Null', 'null', '-', '']

# Replace them with actual missing values
df= df.replace(hidden_nulls, np.nan)

print("Missing values after handling hidden nulls:")
df.isnull().sum().sort_values(ascending=False)

# 6. preprocessing

In [ ]:
# Clean column names by snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print(df.columns.tolist())

In [ ]:
# Strip whitespace and unify casing

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

In [ ]:
# Convert date columns to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Check data types
print(df[date_columns].dtypes)

In [ ]:
# Convert numeric columns to numeric values
numeric_columns = [
    'price',
    'freight_value',
    'payment_value',
    'payment_installments',
    'payment_sequential'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check data types
print(df[numeric_columns].dtypes)

In [ ]:
# Convert all object columns to category
categorical_columns = df.select_dtypes(include='object').columns

for col in categorical_columns:
    df[col] = df[col].astype('category')

print(df.dtypes)

In [ ]:
# Check prices and quantity that are less than 0
print("Prices < 0:")
print((df['price'] < 0).sum())
#it is zero so nothing will change

In [ ]:
# Detect discount scale: 0-1 or 0-100

if 'discount' in df.columns:

    discount_max = df['discount'].max()

    print("Maximum discount value:", discount_max)

    if discount_max <= 1:
        print("Discount scale detected: 0-1")

    elif discount_max <= 100:
        print("Discount scale detected: 0-100")

    else:
        print("Warning: Discount values are outside the expected range.")

# Standardize discount to 0-1 scale

if 'discount' in df.columns:

    if df['discount'].max() > 1:
        df['discount'] = df['discount'] / 100

    print(df['discount'].describe())

In [ ]:
# Handle missing values

#1.drop the rows with missing values in the review_score column

print(df['review_score'].isnull().sum())
df = df.dropna(subset=['review_score'])
print(df['review_score'].isnull().sum())

# 2. Categorical columns → "Unknown"
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_columns:
    if df[col].isna().any():
        if df[col].dtype.name == 'category':
            df[col] = df[col].cat.add_categories(['Unknown'])

        df[col] = df[col].fillna('Unknown')


# 3. Numeric columns → median
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_columns:
  median_value = df[col].median()
  df[col] = df[col].fillna(median_value)

In [ ]:
# Check missing values after handling
print("Missing values after handling:")
print(df.isnull().sum().sum())

In [ ]:
missing = pd.DataFrame({
    'count': df.isna().sum(),
    'percent': df.isna().mean().mul(100).round(2)
}).sort_values('percent', ascending=False).sort_values('count', ascending=False)

print('Missing values after handling')
missing

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_columns:
    df[col] =df[col] = df[col].fillna('Unknown')

print(df.isnull().sum().sum())

In [ ]:
numeric_columns = [
     'price',
    'freight_value',
    'payment_value',
    'payment_installments',
    'payment_sequential'
]
for col in numeric_columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = ((df[col] < lower) | (df[col] > upper)).sum()

        print(f'{col}')
        print(f'Lower limit: {lower:.2f}')
        print(f'Upper limit: {upper:.2f}')
        print(f'Number of outliers: {outliers}')
        print()

#Outlier treatment: Numerical variables were checked using the IQR method. Extreme values were not automatically removed because high prices and
# freight values can represent legitimate e-commerce transactions. Only clearly invalid values, such as non-positive prices, were removed.

# 7. Feature Engeneering

In [ ]:
df["review_score"].value_counts()

# 📅 Date-Based Feature Engineering

To improve the model's ability to predict customer ratings, several time-related features were extracted from the original timestamp columns. These features help capture customer behavior, delivery performance, and seasonal patterns.

## Order Lifecycle & Timestamps

```text
[Customer places order]
       │
       ▼
order_purchase_timestamp
       │
       ▼
[Store approves order]
       │
       ▼
order_approved_at
       │
       ▼
[Seller hands package to carrier]
       │
       ▼
order_delivered_carrier_date
       │
       ▼
[Carrier delivers package to customer]
       │
       ▼
order_delivered_customer_date
       │
       ▼
[Expected delivery date target]
       │
       ▼
order_estimated_delivery_date

## **delivery_days = actual_delivery_date - purchase_date**
The total number of days between the purchase date and the actual delivery date,
This feature measures how long customers had to wait to receive their orders.

In [ ]:
df["delivery_days"] = (
    df["order_delivered_customer_date"]
    - df["order_purchase_timestamp"]
).dt.days

## **expected_delivery_days = estimated_delivery_date - purchase_date**
The number of days the company originally expected the delivery process to take,
This feature represents the company's promised delivery time.

In [ ]:
df["expected_delivery_days"] = (
    df["order_estimated_delivery_date"]
    - df["order_purchase_timestamp"]
).dt.days

## **delivery_delay_days = actual_delivery_days - expected_delivery_days**
The difference between the actual delivery time and the expected delivery time,
This feature directly measures delivery performance.

In [ ]:
df["delivery_delay_days"] = (
    df["delivery_days"]
    - df["expected_delivery_days"]
)

## **is_delayed: A binary feature that indicates whether an order was delivered late**
This feature simplifies delivery performance into a yes/no variable.

In [ ]:
df["is_delayed"] = (df["delivery_delay_days"] > 0).astype(int)

In [ ]:
df["is_delayed"].value_counts()

## Delayed orders received an average rating of 2.50, compared with 4.14 for on-time orders. This suggests that delivery performance is one of the strongest factors affecting customer satisfaction.

In [11]:
df.groupby("is_delayed")["review_score"].mean()

NameError: name 'df' is not defined

## **The month when the order was placed.**
Customer satisfaction may be influenced by seasonal trends.

In [ ]:
df["purchase_month"] = (
    df["order_purchase_timestamp"]
    .dt.month
)

## 0	Monday
## 1	Tuesday
## 2	Wednesday
## 3	Thursday
## 4	Friday
## 5	Saturday
## 6	Sunday

In [ ]:
df["purchase_weekday"] = (
    df["order_purchase_timestamp"]
    .dt.dayofweek
)

## **The hour when the customer placed the order**

In [ ]:
df["purchase_hour"] = (
    df["order_purchase_timestamp"]
    .dt.hour
)

## **A binary feature that indicates whether the order was placed during the weekend**

In [ ]:
df["is_weekend"] = (
    df["purchase_weekday"]
    .isin([5, 6])
    .astype(int)
)

## The day of the week had little effect on customer ratings. Average ratings remained relatively stable across all days, suggesting that purchase timing was not a significant factor affecting customer satisfaction.

In [ ]:
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

(
    df.groupby("purchase_weekday")["review_score"]
    .mean()
    .rename(index=weekday_names)
    .sort_index()
)

## Description

The quarter of the year when the order was placed.

### Values

| Quarter | Months |
| :--- | :--- |
| **Q1** | January – March |
| **Q2** | April – June |
| **Q3** | July – September |
| **Q4** | October – December |

### Business Meaning

This feature captures seasonal patterns that may affect customer satisfaction.

In [ ]:
df["purchase_quarter"] = (
    df["order_purchase_timestamp"]
    .dt.quarter
)

## Customer ratings varied across different quarters of the year. Orders placed during the third quarter (Q3) received the highest average rating (4.18), while orders placed during the first quarter (Q1) received the lowest average rating (3.85). This suggests that seasonal factors may influence customer satisfaction.

In [ ]:
df.groupby("purchase_quarter")["review_score"].mean()

## Delivery status has a significant impact on customer ratings. Successfully delivered orders received an average rating of 4.09, while orders with operational issues received much lower ratings. For example, processing orders had an average rating of 1.41, unavailable orders had 1.59, and canceled orders had 1.91.

In [ ]:
df.groupby("order_status")["review_score"].mean()

## Approval Time (Hours)

**Formula:**
`approval_time_hours` = `order_approved_at` - `order_purchase_timestamp`

### Description
The number of hours between placing the order and its approval.

### Business Meaning
This feature measures how quickly the store processed and approved an order.

In [ ]:
df["approval_time_hours"] = (
    df["order_approved_at"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

## Shipping Preparation Time (Hours)

**Formula:**
`shipping_preparation_hours` = `order_delivered_carrier_date` - `order_approved_at`

### Description
The number of hours between order approval and handing the package over to the shipping carrier.

### Business Meaning
This feature measures the efficiency and speed of fulfillment/warehouse preparation before shipping.

In [ ]:
df["shipping_preparation_hours"] = (
    df["order_delivered_carrier_date"]
    - df["order_approved_at"]
).dt.total_seconds() / 3600

In [ ]:
df[["approval_time_hours", "shipping_preparation_hours"]].describe()

Order approval time showed almost no relationship with customer ratings (correlation = -0.026), suggesting that customers are more sensitive to the overall delivery experience than to the internal approval process.

In [ ]:
df[["approval_time_hours", "review_score"]].corr()

A negative correlation (-0.146) was observed between shipping preparation time and customer ratings. Orders that spent more time in the preparation stage tended to receive lower ratings, indicating that operational delays may negatively affect customer satisfaction.

In [ ]:
df[["shipping_preparation_hours", "review_score"]].corr()

### Product Volume

This feature represents the physical size of the product.

Formula:

product_volume = length × width × height

Larger products may require more handling, packaging, and shipping resources, which can influence the customer experience.

In [ ]:
df["product_volume"] = (
    df["product_length_cm"]
    * df["product_width_cm"]
    * df["product_height_cm"]
)

### Product Density

This feature measures the relationship between a product's weight and its volume.

Formula:

product_density = weight / volume

This feature may help capture differences between lightweight and heavyweight products.

In [ ]:
df["product_density"] = (
    df["product_weight_g"]
    / df["product_volume"]
)

Products with a larger number of images generally received higher customer ratings. Products with only one image had an average rating of 3.97, while products with 5–8 images achieved ratings above 4.15. This suggests that providing multiple product images may help customers better understand product characteristics and set more realistic expectations.

In [ ]:
photos_review = (
    df.groupby("product_photos_qty")
    .agg(
        number_of_products=("product_id", "nunique"),
        average_rating=("review_score", "mean")
    )
    .reset_index()
)

photos_review

### Shipping Price Ratio

This feature measures the proportion of shipping cost relative to the product price.

Formula:

shipping_price_ratio = freight_value / price

High shipping costs relative to product prices may negatively affect customer satisfaction.

In [ ]:
df["shipping_price_ratio"] = (
    df["freight_value"]
    / df["price"]
)

### Average Installment Value

This feature represents the average amount paid in each installment.

Formula:

average_installment_value = payment_value / payment_installments

In [ ]:
df["average_installment_value"] = (
    df["payment_value"]
    / df["payment_installments"]
)

### Total Cost

This feature represents the total amount paid by the customer, including both the product price and the shipping cost.

In [ ]:
df["total_cost"] = (
    df["price"]
    + df["freight_value"]
)

### Delivery Speed Ratio

This feature compares the actual delivery time with the estimated delivery time to measure delivery efficiency.

In [ ]:
df["delivery_speed_ratio"] = (
    df["delivery_days"]
    / df["expected_delivery_days"]
)

### Review Comment Availability

This feature indicates whether the customer wrote a review comment. Customers who leave written feedback may have different satisfaction patterns than those who only provide a rating.

In [ ]:
df["has_review_comment"] = (
    df["review_comment_message"]
    .notna()
    .astype(int)
)

### Multiple Product Photos

This feature indicates whether a product has more than one image. Additional product images may help customers better understand the product before purchasing.

In [ ]:
df["has_multiple_photos"] = (
    df["product_photos_qty"] > 1
).astype(int)

### Multiple Installments

This feature identifies orders that were paid using more than one installment, which may reflect different purchasing behaviors.

In [ ]:
df["multiple_installments"] = (
    df["payment_installments"] > 1
).astype(int)

### Large Product

This feature identifies products with a volume above the dataset median, allowing products to be grouped into large and non-large categories.

In [ ]:
df["large_product"] = (
    df["product_volume"]
    > df["product_volume"].median()
).astype(int)

### Same-State Shipping

This feature indicates whether the customer and the seller are located in the same state. Shorter shipping distances may lead to faster deliveries and higher customer satisfaction.

In [ ]:
df["same_state"] = (
    df["customer_state"].astype(str)
    == df["seller_state"].astype(str)
).astype(int)

In [ ]:
df["same_state"].value_counts()

### Night Purchase

This feature identifies orders placed during late-night hours. Purchasing behavior may vary depending on the time of day.

In [ ]:
df["night_purchase"] = (
    (
        df["purchase_hour"] >= 22
    )
    |
    (
        df["purchase_hour"] <= 6
    )
).astype(int)

### Perfect Rating

In [ ]:
df["perfect_review"] = (
    df["review_score"] == 5
).astype(int)

### Early Delivery

In [ ]:
df["early_delivery"] = (
    df["delivery_delay_days"] < 0
).astype(int)

### Product Size Category

In [ ]:
df["product_size_category"] = pd.qcut(
    df["product_volume"],
    q=4,
    labels=["Small", "Medium", "Large", "Very Large"]
)

### Nmuber Of Products Per Order

In [ ]:
items_per_order = (
    df.groupby("order_id")["order_item_id"]
    .max()
)

df = df.merge(
    items_per_order.rename("items_per_order"),
    on="order_id",
    how="left"
)

In [ ]:
df["items_per_order"].value_counts()

In [ ]:
df.shape

In [ ]:
df

## EDA

## 1. Target Distribution — Rating

 we analyze how customer ratings are distributed from **1 to 5**.

We will check:
- Number of observations for each rating
- Percentage of each rating
- Whether the target is concentrated around certain ratings
- Mean, median and skewness


In [ ]:
# Make sure Rating is numeric and valid

df = df.rename(columns={"review_score": "rating"})

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

rating_missing = df["rating"].isna().sum()
print("Missing rating values:", rating_missing)

# Model 3: drop rows where rating is missing
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"].between(1, 5)].copy()

rating_counts = df["rating"].value_counts().sort_index()
rating_percent = (
    df["rating"].value_counts(normalize=True).sort_index().mul(100).round(2)
)

print("\nRating counts:")
display(rating_counts.rename("count").to_frame())

print("\nRating percentages:")
display(rating_percent.rename("percentage").to_frame())

print(f"\nMean rating: {df['rating'].mean():.2f}")
print(f"Median rating: {df['rating'].median():.2f}")
print(f"Rating skewness: {df['rating'].skew():.3f}")


In [ ]:
# Rating Distribution
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="rating",
    order=sorted(df["rating"].unique())
)
plt.title("Customer Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()


## 2. Feature Distribution

Analyze the main Model 3 numerical variables, where available:

- Age
- Price
- Quantity
- Discount
- Stock
- Delivery Delay
- Payment Value
- Other engineered numerical features


In [ ]:
# Select important numerical features that actually exist in the dataset
feature_candidates = [
    "age",
    "customer_age",
    "price",
    "quantity",
    "discount",
    "discount_percent",
    "stock",
    "stock_before_sale",
    "delivery_delay",
    "payment_value",
    "freight_value",
    "price_gap"
]

available_numeric = [
    c for c in feature_candidates
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

print("Available numerical features:")
print(available_numeric)

for col in available_numeric:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


### EDA Insight — Feature Distribution

For every numerical feature, the histogram helps us understand its range, concentration, and possible outliers.

These distributions are important before regression because:
- strongly skewed variables may need transformation or careful interpretation;
- extreme values can affect Linear Regression;
- scaling will be handled inside the ML pipeline rather than manually before the train/test split.


## 3. Feature vs Target — Rating

we calculate **Mean Rating** for each group.



In [ ]:
def mean_rating_by_group(column, min_count=30):
    if column not in df.columns:
        return None

    result = (
        df.groupby(column, observed=True)["rating"]
          .agg(["mean", "count"])
          .query("count >= @min_count")
          .sort_values("mean", ascending=False)
    )
    return result.rename(columns={"mean": "mean_rating"})

# Category
category_col = next(
    (c for c in ["category", "product_category_name_english"]
     if c in df.columns),
    None
)

if category_col:
    print(f"Mean Rating by {category_col}:")
    display(mean_rating_by_group(category_col).head(15))

# Sales channel
channel_col = next(
    (c for c in ["sales_channel", "channel", "Sales_Channel"]
     if c in df.columns),
    None
)

if channel_col:
    print(f"Mean Rating by {channel_col}:")
    display(mean_rating_by_group(channel_col))

# Age groups
age_col = next(
    (c for c in ["age", "customer_age"]
     if c in df.columns and pd.api.types.is_numeric_dtype(df[c])),
    None
)

if age_col:
    temp = df[[age_col, "rating"]].dropna().copy()
    temp["age_group"] = pd.cut(
        temp[age_col],
        bins=[0, 18, 25, 35, 45, 55, 65, 120],
        labels=["<=18", "19-25", "26-35", "36-45", "46-55", "56-65", "65+"],
        include_lowest=True
    )

    age_rating = (
        temp.groupby("age_group", observed=True)["rating"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "mean_rating"})
    )

    print("Mean Rating by Age Group:")
    display(age_rating)

# Discount groups
discount_col = next(
    (c for c in ["discount_percent", "discount", "discount_percentage"]
     if c in df.columns and pd.api.types.is_numeric_dtype(df[c])),
    None
)

if discount_col:
    temp = df[[discount_col, "rating"]].dropna().copy()
    temp["discount_group"] = pd.qcut(
        temp[discount_col],
        q=5,
        duplicates="drop"
    )

    discount_rating = (
        temp.groupby("discount_group", observed=True)["rating"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "mean_rating"})
    )

    print("Mean Rating by Discount Group:")
    display(discount_rating)

# Stock groups
stock_col = next(
    (c for c in ["stock_before_sale", "stock", "Stock"]
     if c in df.columns and pd.api.types.is_numeric_dtype(df[c])),
    None
)

if stock_col:
    temp = df[[stock_col, "rating"]].dropna().copy()
    temp["stock_group"] = pd.qcut(
        temp[stock_col],
        q=4,
        duplicates="drop"
    )

    stock_rating = (
        temp.groupby("stock_group", observed=True)["rating"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "mean_rating"})
    )

    print("Mean Rating by Stock Group:")
    display(stock_rating)


# Visual: Top 10 Product Categories by Mean Rating

In [ ]:

if category_col:

    # Calculate mean rating + number of ratings per category
    category_plot = (
        df.groupby(category_col, observed=True)["rating"]
          .agg(["mean", "count"])
          .rename(columns={"mean": "mean_rating", "count": "rating_count"})
    )

    # Keep only categories with enough ratings
    category_plot = (
        category_plot[category_plot["rating_count"] >= 100]
        .sort_values("mean_rating", ascending=False)
        .head(10)
        .sort_values("mean_rating", ascending=True)
    )

    # Plot
    plt.figure(figsize=(10, 6))

    bars = plt.barh(
        category_plot.index,
        category_plot["mean_rating"]
    )

    plt.title("Top 10 Product Categories by Mean Rating")
    plt.xlabel("Mean Rating")
    plt.ylabel("Product Category")

    # Rating range
    plt.xlim(1, 5)

    # Add rating value at the end of each bar
    for bar, value in zip(bars, category_plot["mean_rating"]):
        plt.text(
            value + 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            va="center"
        )

    plt.tight_layout()
    plt.show()

# Visual: Mean Rating by Payment Type

In [ ]:


payment_col = "payment_type"

payment_plot = (
    df.groupby(payment_col, observed=True)["rating"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))

sns.barplot(
    x=payment_plot.index,
    y=payment_plot.values
)

plt.title("Mean Rating by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Mean Rating")
plt.ylim(1, 5)
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

# Visual: Mean Rating by Freight Value Group

In [ ]:


temp = df[["freight_value", "rating"]].dropna().copy()

temp["freight_group"] = pd.qcut(
    temp["freight_value"],
    q=5,
    duplicates="drop"
)

freight_rating = (
    temp.groupby("freight_group", observed=True)["rating"]
        .mean()
)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=freight_rating.index.astype(str),
    y=freight_rating.values
)

plt.title("Mean Rating by Freight Value Group")
plt.xlabel("Freight Value Group")
plt.ylabel("Mean Rating")
plt.ylim(1, 5)
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

# Visual: Mean Rating by Product Size Category

In [ ]:


size_col = "product_size_category"

size_rating = (
    df.groupby(size_col, observed=True)["rating"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))

sns.barplot(
    x=size_rating.index,
    y=size_rating.values
)

plt.title("Mean Rating by Product Size Category")
plt.xlabel("Product Size Category")
plt.ylabel("Mean Rating")
plt.ylim(1, 5)
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

We analyze the correlation between `rating` and numerical variables.

The purpose is to identify variables with stronger linear relationships with customer rating.


In [ ]:
corr_candidates = [
    c for c in [
        "rating",
        "age",
        "customer_age",
        "price",
        "quantity",
        "discount",
        "discount_percent",
        "stock",
        "stock_before_sale",
        "delivery_delay",
        "payment_value",
        "freight_value",
        "price_gap"
    ]
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

corr_matrix = df[corr_candidates].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix — Rating and Numerical Features")
plt.tight_layout()
plt.show()

print("Correlation with Rating:")
display(
    corr_matrix["rating"]
    .drop("rating")
    .sort_values(key=abs, ascending=False)
    .to_frame("correlation_with_rating")
)


### EDA Insight — Correlation

- Positive correlation means the variables tend to increase together.
- Negative correlation means higher values of the feature tend to be associated with lower ratings.
- A value close to zero indicates a weak **linear** relationship.
- Correlation does not prove causation.
- For Model 3, this analysis is especially useful for understanding the Linear Regression baseline.


## 5. Time Analysis

If an order date/time column exists, we analyze:
- Average rating over time
- Monthly rating patterns
- Weekday rating patterns


In [ ]:
date_col = next(
    (c for c in [
        "order_date",
        "order_purchase_timestamp",
        "Order_Date"
    ] if c in df.columns),
    None
)

if date_col:
    time_df = df[[date_col, "rating"]].copy()
    time_df[date_col] = pd.to_datetime(time_df[date_col], errors="coerce")
    time_df = time_df.dropna(subset=[date_col])

    time_df["month"] = time_df[date_col].dt.to_period("M").astype(str)
    time_df["weekday"] = time_df[date_col].dt.day_name()

    monthly_rating = time_df.groupby("month")["rating"].mean()

    plt.figure(figsize=(12, 5))
    sns.lineplot(
        x=monthly_rating.index,
        y=monthly_rating.values,
        marker="o"
    )
    plt.title("Average Rating Over Time")
    plt.xlabel("Month")
    plt.ylabel("Mean Rating")
    plt.ylim(1, 5)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    weekday_order = [
        "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday", "Sunday"
    ]

    weekday_rating = (
        time_df.groupby("weekday")["rating"]
        .mean()
        .reindex(weekday_order)
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(
        x=weekday_rating.index,
        y=weekday_rating.values
    )
    plt.title("Average Rating by Purchase Weekday")
    plt.xlabel("Weekday")
    plt.ylabel("Mean Rating")
    plt.ylim(1, 5)
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()
else:
    print("No suitable date column was found; time analysis was skipped.")


### EDA Insight — Time Analysis

- Look for months or weekdays with noticeably higher/lower average ratings.



# Linear Regression

In [ ]:
# Model 3 feature preparation
# Prefer the feature names already created in the shared notebook.
model3_features = [
    "age",
    "customer_age",
    "price",
    "quantity",
    "discount",
    "discount_percent",
    "stock",
    "stock_before_sale",
    "delivery_delay",
    "price_gap",
    "category",
    "product_category_name_english",
    "sales_channel",
    "courier",
    "delivery_status",
    "payment_type"
]

model3_features = [
    c for c in model3_features
    if c in df.columns and c != "rating"
]

X = df[model3_features].copy()
y = df["rating"].astype(float)

print("Model 3 features:")
print(model3_features)
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


In [ ]:
# Linear Regression

linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

model_comparison = pd.DataFrame([{
    "Model": "Linear Regression",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}])

display(model_comparison.round(4))
